In [7]:
import numpy as np
import pandas as pd

data = pd.read_csv('results/emates_results/worker_1/timeseries.csv')
data.head()

,Type,Csid,Cgrid,Cap_kW,waitingLine,Vol_0min,Vol_30min,Vol_60min,ElapsedTime
0,F,90000000,0,0,0,0,0,0,60
1,F,90000001,1,0,0,0,0,0,60
2,F,90000200,0,0,0,0,0,0,60
3,F,90000201,1,0,0,0,0,0,60
4,F,90000400,0,0,0,0,0,0,60


In [30]:
# 通常の辞書アクセス
my_dict = {'a':{'tech' : 'Python', 'version': 3.8}}
#value = my_dict['c']  # KeyError が発生！

# .get() を使った安全なアクセス
value = my_dict.get('c')       # None を返す（エラーなし）
value = my_dict.get('c', 0)    # デフォルト値 0 を返す

print(my_dict.get('a').get('tech'))  # 'Python' を返す

for key, value in my_dict.get('a').items():
    print(key, value)  # 辞書のアイテムを取得

for value in my_dict.get('a', {}).values():
    print(value)  # 辞書の値を取得

Python
tech Python
version 3.8
Python
3.8


In [ ]:
import random
import math

class EVChargerPlacement:
    """
    長期EV充電器配置計画の骨組みアルゴリズム。
    論文「Locating and Sizing Electric Vehicle Chargers Considering Multiple Technologies」の
    逐次増分アルゴリズム（最大フロー問題部分を除く）を簡略化して実装。
    """

    def __init__(self, locations, periods, charging_technologies, demand_data,
                 planning_horizon_years, initial_infrastructure=None,
                 max_chargers_per_location=5,
                 charger_setup_cost={'slow': 20000, 'fast': 100000}, # 技術ごとにコストを設定
                 charger_installation_cost={'slow': 7500, 'fast': 80000}, # 技術ごとにコストを設定
                 charger_capacity_kwh={'slow': 28, 'fast': 300}, # 技術ごとに容量を設定 (kWh/期間)
                 target_coverage_percentage=0.80):
        """
        初期化メソッド。

        Args:
            locations (list): 潜在的な充電ステーション設置候補地のリスト (例: ['LocA', 'LocB', ...])。
                              これがシミュレーションでいう「充電箇所」に対応します。
            periods (list): 需要の時間間隔のリスト (例: ['day', 'night', 'morning', 'afternoon'])。
            charging_technologies (list): 利用可能な充電技術のリスト (例: ['slow', 'fast'])。
            demand_data (dict): 各年、期間、技術、充電箇所ごとの「生まれた需要量」 (kWh)。
                                構造: {year: {period: {technology: {location: demand_kwh}}}}
            planning_horizon_years (int): 計画期間（年数）。
            initial_infrastructure (dict, optional): 計画開始時点の既存インフラ。
                                                     構造: {location: {technology: num_chargers}}。
                                                     デフォルトはNone (既存なし)。
            max_chargers_per_location (int): 各場所で設置可能な充電器の最大数。
            charger_setup_cost (dict): 技術ごとの新規充電ステーション設置コスト。
            charger_installation_cost (dict): 技術ごとの充電器1台あたりの設置コスト。
            charger_capacity_kwh (dict): 技術ごとの充電器1台が1期間に提供できるkWh量。
            target_coverage_percentage (float): 各年で達成すべき需要カバー率の目標 (0.0から1.0)。
        """
        self.locations = locations
        self.periods = periods
        self.charging_technologies = charging_technologies
        self.demand_data = demand_data
        self.planning_horizon_years = planning_horizon_years
        self.max_chargers_per_location = max_chargers_per_location
        self.charger_setup_cost = charger_setup_cost
        self.charger_installation_cost = charger_installation_cost
        self.charger_capacity_kwh = charger_capacity_kwh
        self.target_coverage_percentage = target_coverage_percentage

        # 各場所・技術ごとの現在の設置充電器数
        # {year: {location: {technology: num_chargers}}}
        self.installed_chargers = {}
        # 各場所・技術ごとの設置済みフラグ (セットアップコストを支払ったか)
        # {year: {location: {technology: bool}}}
        self.setup_locations = {}

        # 初期インフラのセットアップ
        for year in range(1, self.planning_horizon_years + 1):
            self.installed_chargers[year] = {loc: {tech: 0 for tech in charging_technologies} for loc in locations}
            self.setup_locations[year] = {loc: {tech: False for tech in charging_technologies} for loc in locations}

        if initial_infrastructure:
            for loc, techs in initial_infrastructure.items():
                if loc in self.locations:
                    for tech, num in techs.items():
                        if tech in self.charging_technologies:
                            self.installed_chargers[1][loc][tech] = num
                            if num > 0:
                                self.setup_locations[1][loc][tech] = True

    def _get_current_chargers(self, year, location, technology):
        """
        指定された年までの累積充電器数を取得するヘルパー関数。
        """
        total_chargers = 0
        for y in range(1, year + 1):
            total_chargers += self.installed_chargers[y].get(location, {}).get(technology, 0)
        return total_chargers

    def _is_location_setup(self, year, location, technology):
        """
        指定された年までに場所がセットアップされているかを確認するヘルパー関数。
        """
        for y in range(1, year + 1):
            if self.setup_locations[y].get(location, {}).get(technology, False):
                return True
        return False

    def _calculate_demand_coverage(self, year, current_chargers_config, current_year_demand_data, total_requested_demand):
        """
        現在の充電器配置に基づく需要カバー率を計算する関数。
        シミュレーションから得られる「生まれた需要」と、設置された充電器の供給能力を比較します。

        Args:
            year (int): 現在の計画年。
            current_chargers_config (dict): その年までの累積充電器配置
                                            {location: {technology: num_chargers}}。
            current_year_demand_data (dict): その年の「生まれた需要」データ {period: {technology: {location: demand_kwh}}}。
            total_requested_demand (float): その年の全ての期間・技術・場所における「生まれた需要」の総和。

        Returns:
            float: 計算された需要カバー率 (0.0から1.0)。
        """
        
        if total_requested_demand == 0:
            return 1.0 # 生まれた需要がなければ100%カバー

        covered_demand_kwh = 0
        
        for period in self.periods:
            for tech in self.charging_technologies:
                for loc in self.locations:
                    # その場所・技術・期間の「生まれた需要」
                    requested_demand_at_loc_tech_period = current_year_demand_data.get(period, {}).get(tech, {}).get(loc, 0)
                    
                    # その場所・技術の充電器数（累積）
                    num_chargers = current_chargers_config.get(loc, {}).get(tech, 0)
                    
                    # その場所・技術・期間の供給可能容量 (充電器のタイプ別容量を使用)
                    supply_capacity_kwh = num_chargers * self.charger_capacity_kwh.get(tech, 0) # 辞書から取得
                    
                    # 「生まれた需要」と供給可能容量の小さい方が、実際にカバーされた需要量
                    covered_demand_kwh += min(requested_demand_at_loc_tech_period, supply_capacity_kwh)

        # 総「生まれた需要」に対するカバー率
        estimated_coverage = min(1.0, covered_demand_kwh / total_requested_demand)
        
        # 実際のシミュレーションの「充電達成率」を模倣するため、少しノイズを加える
        return estimated_coverage * random.uniform(0.95, 1.0) # わずかな変動をシミュレート


    def _calculate_cost(self, num_new_chargers, location, technology, year):
        """
        充電器の設置・増設にかかるコストを計算する。
        セットアップコストは、その場所・技術で初めて設置する場合にのみ発生。
        """
        cost = num_new_chargers * self.charger_installation_cost.get(technology, 0) # 技術別コスト
        if not self._is_location_setup(year, location, technology):
            cost += self.charger_setup_cost.get(technology, 0) # 技術別コスト
        return cost

    def run_planning(self):
        """
        長期充電配置計画を実行する。
        """
        print(f"--- EV充電器長期配置計画を開始 ---")
        print(f"計画期間: {self.planning_horizon_years}年")
        print(f"目標カバー率: {self.target_coverage_percentage * 100:.1f}%")
        print("-" * 40)

        total_investment_cost = 0

        for year in range(1, self.planning_horizon_years + 1):
            print(f"\n=== 年 {year} の計画 ===\n")

            # 前年までの充電器配置をコピーし、その年の初期状態とする
            if year > 1:
                for loc in self.locations:
                    for tech in self.charging_technologies:
                        self.installed_chargers[year][loc][tech] = self._get_current_chargers(year - 1, loc, tech)
                        self.setup_locations[year][loc][tech] = self._is_location_setup(year - 1, loc, tech)

            # その年の総「生まれた需要」を計算（demand_dataから取得）
            current_year_demand_data = self.demand_data.get(year, {})
            if not current_year_demand_data:
                print(f"警告: 年 {year} の需要データが見つかりません。この年の計画はスキップされます。")
                continue

            total_requested_demand = 0
            for period_data in current_year_demand_data.values():
                for tech_data in period_data.values():
                    total_requested_demand += sum(tech_data.values())
            
            if total_requested_demand == 0:
                print(f"警告: 年 {year} の総「生まれた需要」が0です。この年の計画はスキップされます。")
                continue

            print(f"年 {year} の総「生まれた需要」: {total_requested_demand:.2f} kWh")

            # 現在の充電器配置に基づく初期カバー率を計算
            current_coverage = self._calculate_demand_coverage(
                year, self.installed_chargers[year], current_year_demand_data, total_requested_demand
            )
            print(f"初期需要カバー率 (充電達成率): {current_coverage * 100:.2f}%")

            year_investment_cost = 0
            iteration = 0

            # 目標カバー率に達するまで充電器を増設
            while current_coverage < self.target_coverage_percentage:
                iteration += 1
                print(f"  --- 設置試行 {iteration} ---")

                best_candidate = None
                max_value_per_cost = -1.0

                # 論文の「2段階アプローチ」を簡略化して実装
                # 各候補地と技術について、最も効率の良い増設を探索
                for loc in self.locations:
                    for tech in self.charging_technologies:
                        current_chargers_at_loc_tech = self._get_current_chargers(year, loc, tech)
                        
                        # 設置可能な充電器の最大数まで試行
                        # 論文ではM_j^kが上限
                        remaining_capacity = self.max_chargers_per_location - current_chargers_at_loc_tech

                        if remaining_capacity <= 0:
                            continue # これ以上設置できない

                        # 1台から残りの容量まで試行
                        for num_to_install in range(1, remaining_capacity + 1):
                            # 一時的に充電器を追加した場合の容量増加をシミュレート
                            temp_chargers_config = {
                                l: {t: self._get_current_chargers(year, l, t) for t in self.charging_technologies}
                                for l in self.locations
                            }
                            temp_chargers_config[loc][tech] += num_to_install

                            # 新しい配置でのカバー率を計算
                            new_coverage = self._calculate_demand_coverage(
                                year, temp_chargers_config, current_year_demand_data, total_requested_demand
                            )
                            
                            # カバー率の増加量
                            coverage_increase = new_coverage - current_coverage

                            if coverage_increase <= 0:
                                continue # カバー率が増加しない場合はスキップ

                            # 設置にかかるコストを計算
                            cost_of_installation = self._calculate_cost(num_to_install, loc, tech, year)

                            if cost_of_installation == 0: # コストが0の場合は無限大の効率と見なす
                                value_per_cost = float('inf')
                            else:
                                value_per_cost = coverage_increase / cost_of_installation

                            # 最も効率の良い候補を選択
                            if value_per_cost > max_value_per_cost:
                                max_value_per_cost = value_per_cost
                                best_candidate = (loc, tech, num_to_install, cost_of_installation, new_coverage)

                if best_candidate:
                    loc, tech, num_to_install, cost_incurred, new_coverage = best_candidate
                    
                    # 最適な候補を実際に設置
                    self.installed_chargers[year][loc][tech] += num_to_install
                    if not self._is_location_setup(year, loc, tech):
                        self.setup_locations[year][loc][tech] = True
                    
                    year_investment_cost += cost_incurred
                    total_investment_cost += cost_incurred
                    current_coverage = new_coverage

                    print(f"    選択: 場所 '{loc}', 技術 '{tech}', 充電器数 {num_to_install}台")
                    print(f"    追加コスト: {cost_incurred:,.2f}円")
                    print(f"    現在の需要カバー率 (充電達成率): {current_coverage * 100:.2f}%")
                else:
                    print("    これ以上充電器を設置しても目標カバー率に達しないか、効率的な設置が見つかりませんでした。")
                    break # これ以上改善が見られない場合はループを抜ける

            print(f"\n年 {year} の最終需要カバー率 (充電達成率): {current_coverage * 100:.2f}%")
            print(f"年 {year} の総投資コスト: {year_investment_cost:,.2f}円")
            print(f"年 {year} 末の充電器配置:")
            for loc in self.locations:
                installed_at_loc = {tech: self._get_current_chargers(year, loc, tech) for tech in self.charging_technologies}
                if any(installed_at_loc.values()):
                    print(f"  {loc}: {installed_at_loc}")
            print("-" * 40)

        print(f"\n--- EV充電器長期配置計画が完了しました ---")
        print(f"総投資コスト: {total_investment_cost:,.2f}円")
        print("\n最終的な充電器配置（各年末時点の累積）:")
        for year in range(1, self.planning_horizon_years + 1):
            print(f"  年 {year}:")
            for loc in self.locations:
                installed_at_loc = {tech: self._get_current_chargers(year, loc, tech) for tech in self.charging_technologies}
                if any(installed_at_loc.values()):
                    print(f"    {loc}: {installed_at_loc}")

# --- 使用例 ---
if __name__ == "__main__":
    # サンプルデータ
    locations = ['StationA', 'StationB', 'StationC', 'StationD', 'StationE']
    periods = ['morning', 'afternoon', 'evening'] # シミュレーションの間隔
    charging_technologies = ['slow', 'fast']

    # 論文のパラメータに基づくコストと容量
    charger_setup_costs = {'slow': 20000, 'fast': 100000}
    charger_installation_costs = {'slow': 7500, 'fast': 80000}
    # 論文では「kWh/day」とある。期間は一日と仮定し、充電器の容量を設定。
    # ここでは，充電器出力を50, 100kWと設定する。
    charger_capacities_kwh = {'slow': 1200, 'fast': 2400} 

    # ダミーの「生まれた需要」データ（各年、期間、技術、充電箇所ごとの需要）
    # あなたのシミュレーションから、この形式で「充電を希望した総量」を抽出・集計してください
    demand_data_for_planning = {
        1: { # 年 1
            'morning': {
                'slow': {'StationA': 100, 'StationB': 80, 'StationC': 120, 'StationD': 50, 'StationE': 70},
                'fast': {'StationA': 30, 'StationB': 60, 'StationC': 40, 'StationD': 20, 'StationE': 35},
            },
            'afternoon': {
                'slow': {'StationA': 150, 'StationB': 100, 'StationC': 180, 'StationD': 70, 'StationE': 90},
                'fast': {'StationA': 40, 'StationB': 80, 'StationC': 50, 'StationD': 25, 'StationE': 45},
            },
            'evening': {
                'slow': {'StationA': 200, 'StationB': 120, 'StationC': 250, 'StationD': 90, 'StationE': 110},
                'fast': {'StationA': 50, 'StationB': 100, 'StationC': 60, 'StationD': 30, 'StationE': 55},
            },
        },
        2: { # 年 2 (需要が5%成長すると仮定)
            'morning': {
                'slow': {'StationA': 105, 'StationB': 84, 'StationC': 126, 'StationD': 52.5, 'StationE': 73.5},
                'fast': {'StationA': 31.5, 'StationB': 63, 'StationC': 42, 'StationD': 21, 'StationE': 36.75},
            },
            'afternoon': {
                'slow': {'StationA': 157.5, 'StationB': 105, 'StationC': 189, 'StationD': 73.5, 'StationE': 94.5},
                'fast': {'StationA': 42, 'StationB': 84, 'StationC': 52.5, 'StationD': 26.25, 'StationE': 47.25},
            },
            'evening': {
                'slow': {'StationA': 210, 'StationB': 126, 'StationC': 262.5, 'StationD': 94.5, 'StationE': 115.5},
                'fast': {'StationA': 52.5, 'StationB': 105, 'StationC': 63, 'StationD': 31.5, 'StationE': 57.75},
            },
        },
        3: { # 年 3 (需要がさらに5%成長すると仮定)
            'morning': {
                'slow': {'StationA': 110.25, 'StationB': 88.2, 'StationC': 132.3, 'StationD': 55.125, 'StationE': 77.175},
                'fast': {'StationA': 33.075, 'StationB': 66.15, 'StationC': 44.1, 'StationD': 22.05, 'StationE': 38.5875},
            },
            'afternoon': {
                'slow': {'StationA': 165.375, 'StationB': 110.25, 'StationC': 198.45, 'StationD': 77.175, 'StationE': 99.225},
                'fast': {'StationA': 44.1, 'StationB': 88.2, 'StationC': 55.125, 'StationD': 27.5625, 'StationE': 49.6125},
            },
            'evening': {
                'slow': {'StationA': 220.5, 'StationB': 132.3, 'StationC': 275.625, 'StationD': 99.225, 'StationE': 121.275},
                'fast': {'StationA': 55.125, 'StationB': 110.25, 'StationC': 66.15, 'StationD': 33.075, 'StationE': 60.6375},
            },
        },
    }

    # 初期インフラの例 (オプション)
    initial_infra = {
        'StationA': {'slow': 1},
        'StationC': {'fast': 1}
    }

    # プランナーのインスタンス化と実行
    planner = EVChargerPlacement(
        locations=locations,
        periods=periods,
        charging_technologies=charging_technologies,
        demand_data=demand_data_for_planning,
        planning_horizon_years=3,
        initial_infrastructure=initial_infra,
        max_chargers_per_location=5,
        charger_setup_cost=charger_setup_costs,
        charger_installation_cost=charger_installation_costs,
        charger_capacity_kwh=charger_capacities_kwh,
        target_coverage_percentage=0.90 # 目標カバー率を高く設定
    )
    planner.run_planning()


--- EV充電器長期配置計画を開始 ---
計画期間: 3年
目標カバー率: 90.0%
----------------------------------------

=== 年 1 の計画 ===

年 1 の総「生まれた需要」: 2500.00 kWh
初期需要カバー率 (充電達成率): 9.26%
  --- 設置試行 1 ---
    選択: 場所 'StationA', 技術 'slow', 充電器数 1台
    追加コスト: 7,500.00円
    現在の需要カバー率 (充電達成率): 12.65%
  --- 設置試行 2 ---
    選択: 場所 'StationA', 技術 'slow', 充電器数 2台
    追加コスト: 15,000.00円
    現在の需要カバー率 (充電達成率): 18.95%
  --- 設置試行 3 ---
    選択: 場所 'StationA', 技術 'slow', 充電器数 1台
    追加コスト: 7,500.00円
    現在の需要カバー率 (充電達成率): 21.20%
  --- 設置試行 4 ---
    選択: 場所 'StationC', 技術 'slow', 充電器数 5台
    追加コスト: 57,500.00円
    現在の需要カバー率 (充電達成率): 35.91%
  --- 設置試行 5 ---
    選択: 場所 'StationB', 技術 'slow', 充電器数 4台
    追加コスト: 50,000.00円
    現在の需要カバー率 (充電達成率): 47.72%
  --- 設置試行 6 ---
    選択: 場所 'StationE', 技術 'slow', 充電器数 4台
    追加コスト: 50,000.00円
    現在の需要カバー率 (充電達成率): 59.44%
  --- 設置試行 7 ---
    選択: 場所 'StationD', 技術 'slow', 充電器数 2台
    追加コスト: 35,000.00円
    現在の需要カバー率 (充電達成率): 65.57%
  --- 設置試行 8 ---
    選択: 場所 'StationD', 技術 'slow', 充電器数 1台
    追加コスト

In [ ]:
import random
import math

class EVChargerPlacement:
    """
    長期EV充電器配置計画の骨組みアルゴリズム。
    論文「Locating and Sizing Electric Vehicle Chargers Considering Multiple Technologies」の
    逐次増分アルゴリズム（最大フロー問題部分を除く）を簡略化して実装。
    """

    def __init__(self, locations, demand_zones, planning_horizon_years,
                 charging_technologies, initial_infrastructure=None,
                 max_chargers_per_location=5, charger_setup_cost=20000,
                 charger_installation_cost=7500, charger_capacity_kwh=28,
                 target_coverage_percentage=0.80):
        """
        初期化メソッド。

        Args:
            locations (list): 潜在的な充電ステーション設置候補地のリスト (例: ['LocA', 'LocB', ...])。
            demand_zones (list): 需要地のリスト (例: ['Zone1', 'Zone2', ...])。
            planning_horizon_years (int): 計画期間（年数）。
            demand_data (dict): 各年、期間、技術、充電箇所ごとの需要量 (kWh)。
                    構造: {year: {period: {technology: {location: demand_kwh}}}}
            charging_technologies (list): 利用可能な充電技術のリスト (例: ['slow', 'fast'])。
            initial_infrastructure (dict, optional): 計画開始時点の既存インフラ。
                                                     {location: {technology: num_chargers}} の形式。
                                                     デフォルトはNone (既存なし)。
            max_chargers_per_location (int): 各場所で設置可能な充電器の最大数。
            charger_setup_cost (float): 新しい充電ステーションを設置する際の固定コスト。
            charger_installation_cost (float): 充電器1台あたりの設置コスト。
            charger_capacity_kwh (float): 充電器1台が1期間に提供できるkWh量。
            target_coverage_percentage (float): 各年で達成すべき需要カバー率の目標 (0.0から1.0)。
        """
        self.locations = locations
        self.demand_zones = demand_zones
        self.planning_horizon_years = planning_horizon_years
        self.charging_technologies = charging_technologies
        self.max_chargers_per_location = max_chargers_per_location
        self.charger_setup_cost = charger_setup_cost
        self.charger_installation_cost = charger_installation_cost
        self.charger_capacity_kwh = charger_capacity_kwh
        self.target_coverage_percentage = target_coverage_percentage

        # 各場所・技術ごとの現在の設置充電器数
        # {year: {location: {technology: num_chargers}}}
        self.installed_chargers = {}
        # 各場所・技術ごとの設置済みフラグ (セットアップコストを支払ったか)
        # {year: {location: {technology: bool}}}
        self.setup_locations = {}

        # 初期インフラのセットアップ
        """年度ごとの初期値の設定。構造：{場所: {技術: 充電器数}}
        データ構造の詳細：
        第1層: year - 計画年（例：1, 2, 3）
        第2層: loc - 設置候補地（例：'A', 'B', 'C', 'D', 'E'）=>csPositonをもとに設定。
        第3層: tech - 充電技術（例：'slow', 'fast'）=>
        値: False - 初期状態では全てセットアップ未完了
        """
        for year in range(1, self.planning_horizon_years + 1):
            self.installed_chargers[year] = {loc: {tech: 0 for tech in charging_technologies} for loc in locations}
            self.setup_locations[year] = {loc: {tech: False for tech in charging_technologies} for loc in locations}

        if initial_infrastructure:
            for loc, techs in initial_infrastructure.items():
                if loc in self.locations:
                    for tech, num in techs.items():
                        if tech in self.charging_technologies:
                            self.installed_chargers[1][loc][tech] = num
                            if num > 0:
                                self.setup_locations[1][loc][tech] = True

    def _get_current_chargers(self, year, location, technology):
        """
        指定された年までの累積充電器数を取得するヘルパー関数。
        """
        total_chargers = 0
        for y in range(1, year + 1):
            total_chargers += self.installed_chargers[y].get(location, {}).get(technology, 0)
        return total_chargers

    def _is_location_setup(self, year, location, technology):
        """
        指定された年までに場所がセットアップされているかを確認するヘルパー関数。
        """
        for y in range(1, year + 1):
            if self.setup_locations[y].get(location, {}).get(technology, False):
                return True
        return False

    def _calculate_demand_coverage(self, year, current_chargers_config, total_demand):
        """
        現在の充電器配置に基づく需要カバー率を計算するダミー関数。
        論文の最大フロー問題の代替として、ここでは簡略化されたロジックを使用。
        充電器数が多いほどカバー率が高くなると仮定。
        
        本研究では，外部シミュレーションを用いて，乱数的に充電する車両の数・時間を決めているため，事前定義が困難
        そのため，潜在充電量の代わりにシミュレーション後の充電達成率を導入して，

        Args:
            year (int): 現在の計画年。
            current_chargers_config (dict): その年までの累積充電器配置
                                            {location: {technology: num_chargers}}。
            total_demand (float): その年の総需要。

        Returns:
            float: 計算された需要カバー率 (0.0から1.0)。
        """
        if total_demand == 0:
            return 1.0 # 需要がなければ100%カバー

        # 単純なカバー率計算ロジック:
        # 設置されている充電器の総容量と、場所の数に基づいてカバー率を推定
        total_installed_capacity = 0
        for loc in self.locations:
            for tech in self.charging_technologies:
                num_chargers = current_chargers_config.get(loc, {}).get(tech, 0)
                total_installed_capacity += num_chargers * self.charger_capacity_kwh

        # カバー率は総容量に比例し、総需要との比率で正規化
        # 論文の「predetermined adequate level of demand coverage」を模倣
        # ここでは、総容量が総需要の何倍かによってカバー率を決定する
        # 例えば、総容量が総需要の1倍なら50%カバー、2倍なら80%カバー、3倍なら95%カバーといった線形/非線形関係
        # ここでは非常に単純に、総容量が多いほどカバー率が高いとする
        # 実際の需要カバーは、需要地の分布、充電ステーションへの距離、時間帯などを考慮する必要があるが、
        # 今回は「骨組み」なので簡略化。
        
        # 例: 総容量が総需要のX%に達すると、カバー率もX%に達すると仮定
        # ただし、最大は1.0
        estimated_coverage = min(1.0, total_installed_capacity / total_demand)
        
        # 論文の「target level of demand coverage」を達成するために、
        # 実際のカバー率を少しランダムに変動させることで、逐次的な設置の必要性をシミュレート
        return estimated_coverage * random.uniform(0.9, 1.1) # 実際のカバー率に少しノイズを加える

    def _generate_demand_data(self, year):
        """
        年ごとのダミー需要データを生成する。
        実際のアプリケーションでは、これは外部から提供される。
        ーーー追記ーーー
        raed_csvで保存しているファイルから結果を取得（もしくはメモリ上に保有）し，需要を返す
        """
        # 各需要地、技術、期間（日中/夜間）ごとの需要をシミュレート
        # 簡単のため、ここでは総需要のみを返す
        base_demand_per_zone = 1000 # kWh/年
        # 年が進むにつれて需要が増加すると仮定
        demand_growth_rate = 0.05
        total_demand = len(self.demand_zones) * base_demand_per_zone * (1 + demand_growth_rate)**(year - 1)
        return total_demand

    def _calculate_cost(self, num_new_chargers, location, technology, year):
        """
        充電器の設置・増設にかかるコストを計算する。
        セットアップコストは、その場所・技術で初めて設置する場合にのみ発生。
        ーーー追記ーーー
        
        
        """
        cost = num_new_chargers * self.charger_installation_cost
        if not self._is_location_setup(year, location, technology):
            cost += self.charger_setup_cost
        return cost

    def run_planning(self):
        """
        長期充電配置計画を実行する。
        """
        print(f"--- EV充電器長期配置計画を開始 ---")
        print(f"計画期間: {self.planning_horizon_years}年")
        print(f"目標カバー率: {self.target_coverage_percentage * 100:.1f}%")
        print("-" * 40)

        total_investment_cost = 0

        for year in range(1, self.planning_horizon_years + 1):
            print(f"\n=== 年 {year} の計画 ===\n")

            # 前年までの充電器配置をコピーし、その年の初期状態とする
            if year > 1:
                for loc in self.locations:
                    for tech in self.charging_technologies:
                        self.installed_chargers[year][loc][tech] = self._get_current_chargers(year - 1, loc, tech)
                        self.setup_locations[year][loc][tech] = self._is_location_setup(year - 1, loc, tech)

            # その年の総需要を生成（または取得）
            current_year_total_demand = self._generate_demand_data(year)
            print(f"年 {year} の総需要: {current_year_total_demand:.2f} kWh")

            current_coverage = self._calculate_demand_coverage(year, self.installed_chargers[year], current_year_total_demand)
            print(f"初期需要カバー率: {current_coverage * 100:.2f}%")

            year_investment_cost = 0
            iteration = 0

            # 目標カバー率に達するまで充電器を増設
            while current_coverage < self.target_coverage_percentage:
                iteration += 1
                print(f"  --- 設置試行 {iteration} ---")

                best_candidate = None
                max_value_per_cost = -1.0

                # 論文の「2段階アプローチ」を簡略化して実装
                # 各候補地と技術について、最も効率の良い増設を探索
                for loc in self.locations:
                    for tech in self.charging_technologies:
                        current_chargers_at_loc_tech = self._get_current_chargers(year, loc, tech)
                        
                        # 設置可能な充電器の最大数まで試行
                        # 論文ではM_j^kが上限
                        remaining_capacity = self.max_chargers_per_location - current_chargers_at_loc_tech

                        if remaining_capacity <= 0:
                            continue # これ以上設置できない

                        # 1台から残りの容量まで試行
                        for num_to_install in range(1, remaining_capacity + 1):
                            # 一時的に充電器を追加した場合の容量増加
                            temp_chargers_config = {
                                l: {t: self._get_current_chargers(year, l, t) for t in self.charging_technologies}
                                for l in self.locations
                            }
                            temp_chargers_config[loc][tech] += num_to_install

                            # 論文のdelta_j^pk (追加でカバーできる需要量) を模倣
                            # ここでは、追加容量がカバー率にどれだけ貢献するかを評価
                            # 論文では最大フロー問題で正確に計算するが、ここでは簡易的に
                            
                            # 新しい配置でのカバー率を計算
                            new_coverage = self._calculate_demand_coverage(year, temp_chargers_config, current_year_total_demand)
                            
                            # カバー率の増加量
                            coverage_increase = new_coverage - current_coverage

                            if coverage_increase <= 0:
                                continue # カバー率が増加しない場合はスキップ

                            # 設置にかかるコストを計算
                            cost_of_installation = self._calculate_cost(num_to_install, loc, tech, year)

                            if cost_of_installation == 0: # コストが0の場合は無限大の効率と見なす
                                value_per_cost = float('inf')
                            else:
                                value_per_cost = coverage_increase / cost_of_installation

                            # 最も効率の良い候補を選択
                            if value_per_cost > max_value_per_cost:
                                max_value_per_cost = value_per_cost
                                best_candidate = (loc, tech, num_to_install, cost_of_installation, new_coverage)

                if best_candidate:
                    loc, tech, num_to_install, cost_incurred, new_coverage = best_candidate
                    
                    # 最適な候補を実際に設置
                    self.installed_chargers[year][loc][tech] += num_to_install
                    if not self._is_location_setup(year, loc, tech):
                        self.setup_locations[year][loc][tech] = True
                    
                    year_investment_cost += cost_incurred
                    total_investment_cost += cost_incurred
                    current_coverage = new_coverage

                    print(f"    選択: 場所 '{loc}', 技術 '{tech}', 充電器数 {num_to_install}台")
                    print(f"    追加コスト: {cost_incurred:,.2f}円")
                    print(f"    現在の需要カバー率: {current_coverage * 100:.2f}%")
                else:
                    print("    これ以上充電器を設置しても目標カバー率に達しないか、効率的な設置が見つかりませんでした。")
                    break # これ以上改善が見られない場合はループを抜ける

            print(f"\n年 {year} の最終需要カバー率: {current_coverage * 100:.2f}%")
            print(f"年 {year} の総投資コスト: {year_investment_cost:,.2f}円")
            print(f"年 {year} 末の充電器配置:")
            for loc in self.locations:
                installed_at_loc = {tech: self._get_current_chargers(year, loc, tech) for tech in self.charging_technologies}
                if any(installed_at_loc.values()):
                    print(f"  {loc}: {installed_at_loc}")
            print("-" * 40)

        print(f"\n--- EV充電器長期配置計画が完了しました ---")
        print(f"総投資コスト: {total_investment_cost:,.2f}円")
        print("\n最終的な充電器配置（各年末時点の累積）:")
        for year in range(1, self.planning_horizon_years + 1):
            print(f"  年 {year}:")
            for loc in self.locations:
                installed_at_loc = {tech: self._get_current_chargers(year, loc, tech) for tech in self.charging_technologies}
                if any(installed_at_loc.values()):
                    print(f"    {loc}: {installed_at_loc}")

# --- 使用例 ---
if __name__ == "__main__":
    # サンプルデータ
    locations = ['A', 'B', 'C', 'D', 'E']
    demand_zones = ['Z1', 'Z2', 'Z3', 'Z4', 'Z5', 'Z6', 'Z7', 'Z8', 'Z9', 'Z10']
    planning_horizon_years = 3
    charging_technologies = ['slow', 'fast']

    # 初期インフラの例 (オプション)
    initial_infra = {
        'A': {'slow': 1},
        'C': {'fast': 1}
    }

    # プランナーのインスタンス化と実行
    planner = EVChargerPlacement(
        locations=locations,
        demand_zones=demand_zones,
        planning_horizon_years=planning_horizon_years,
        charging_technologies=charging_technologies,
        initial_infrastructure=initial_infra,
        max_chargers_per_location=5,
        charger_setup_cost=500000,  # 新規設置コストを少し高く設定
        charger_installation_cost=100000, # 1台あたりの設置コスト
        charger_capacity_kwh=300, # 充電器1台あたりの容量を高く設定
        target_coverage_percentage=0.90 # 目標カバー率を高く設定
    )
    planner.run_planning()


--- EV充電器長期配置計画を開始 ---
計画期間: 3年
目標カバー率: 90.0%
----------------------------------------

=== 年 1 の計画 ===

年 1 の総需要: 10000.00 kWh
初期需要カバー率: 5.96%
  --- 設置試行 1 ---
    選択: 場所 'C', 技術 'fast', 充電器数 2台
    追加コスト: 200,000.00円
    現在の需要カバー率: 12.93%
  --- 設置試行 2 ---
    選択: 場所 'A', 技術 'slow', 充電器数 4台
    追加コスト: 400,000.00円
    現在の需要カバー率: 25.61%
  --- 設置試行 3 ---
    選択: 場所 'C', 技術 'fast', 充電器数 2台
    追加コスト: 200,000.00円
    現在の需要カバー率: 32.39%
  --- 設置試行 4 ---
    選択: 場所 'C', 技術 'slow', 充電器数 5台
    追加コスト: 1,000,000.00円
    現在の需要カバー率: 48.58%
  --- 設置試行 5 ---
    選択: 場所 'E', 技術 'slow', 充電器数 4台
    追加コスト: 900,000.00円
    現在の需要カバー率: 62.31%
  --- 設置試行 6 ---
    選択: 場所 'B', 技術 'fast', 充電器数 5台
    追加コスト: 1,000,000.00円
    現在の需要カバー率: 78.39%
  --- 設置試行 7 ---
    選択: 場所 'D', 技術 'fast', 充電器数 5台
    追加コスト: 1,000,000.00円
    現在の需要カバー率: 95.10%

年 1 の最終需要カバー率: 95.10%
年 1 の総投資コスト: 4,700,000.00円
年 1 末の充電器配置:
  A: {'slow': 5, 'fast': 0}
  B: {'slow': 0, 'fast': 5}
  C: {'slow': 5, 'fast': 5}
  D: {'slow': 0, 'fast':